# 05e: Computational efficiency, figures and tables

**Paper:** Section *Computational Efficiency*

In [ ]:
import json
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
import numpy as np

RESULTS_DIR = Path("data/efficiency")
SRC = RESULTS_DIR / "results.jsonl"
OUT = Path("figures/5-efficiency")
OUT.mkdir(parents=True, exist_ok=True)

BLUE, ORANGE, INK, INK2, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e4e3df"
H_COLORS = {17: "#86b6ef", 65: "#2a78d6", 257: "#104281"}  # ordinal blue ramp
plt.rcParams.update({"font.size": 8, "axes.edgecolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
                     "axes.labelcolor": INK, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "pdf.fonttype": 42,
                     "legend.frameon": False})

## Load the results

In [ ]:
recs = [json.loads(line) for line in open(SRC)]
env = next(r for r in recs if r["kind"] == "env")
diwect = {(r["complex"], r["h"], r["dtype"]): r for r in recs if r["kind"] == "diwect" and not r.get("oom")}
# DIWECT re-timed with the calibrated chunk sizes (05b with DIWECT_ONLY = True), incl. cubical complexes
RETIMED = RESULTS_DIR / "results_diwect.jsonl"
if RETIMED.exists():
    retimed = [json.loads(line) for line in open(RETIMED)]
    diwect = {(r["complex"], r["h"], r["dtype"]): r for r in retimed if r["kind"] == "diwect" and not r.get("oom")}
    chunk = next(r for r in retimed if r["kind"] == "env").get("chunk")
    print(f"DIWECT timings from {RETIMED.name} (chunk sizes {chunk})")
wect = {}
for r in recs:
    if r["kind"] == "wect" and not r.get("oom"):
        wect.setdefault((r["complex"], r["sampling"]), []).append(r)
cells = {r["complex"]: r["cells"] for r in recs if "complex" in r}
meta = {r["complex"]: r for r in recs if "complex" in r}
HEIGHTS = (17, 65, 257)


def label(name):
    m = meta[name]
    if m["source"] == "barley seed":
        kind = {"mesh": "seed surface mesh", "freudenthal": "seed voxels, triangulated",
                "cubical": "seed voxels, cubical"}[m["ctype"]]
    else:
        kind = f'{m["source"]}, {"triangulated" if m["ctype"] == "freudenthal" else m["ctype"]}'
    return f"{kind} ({cells[name]:,} cells)"


def d_for(name, h, tol):
    """Smallest d (pyECT's directions) whose average is within tol; None if never reached."""
    for r in sorted(wect.get((name, "grid"), []), key=lambda r: r["d"]):
        if r["rel_err"][str(h)] <= tol:
            return r
    return None

## Figure: convergence of the averaged WECT to the DIWECT

In [ ]:
# ---------------------------------------------------------------------------
PANELS = [n for n in ("img512_bin_freudenthal", "img512_gray_freudenthal", "seed_mesh_step1",
                      "seed_vox2_freudenthal") if (n, "grid") in wect]
if not PANELS:
    PANELS = sorted({n for n, s in wect if meta[n]["ctype"] != "cubical"}, key=cells.get)[:4]

fig, axes = plt.subplots(1, len(PANELS), figsize=(5.9, 2.3), sharey=True)
for ax, name in zip(np.atleast_1d(axes), PANELS):
    for h in HEIGHTS:
        rows = sorted(wect[name, "grid"], key=lambda r: r["d"])
        ax.plot([r["d"] for r in rows], [max(r["rel_err"][str(h)], 1e-9) for r in rows], color=H_COLORS[h],
                lw=1.5, marker="o", ms=2.5, label=f"WECT average, $h = {h}$")
    rows = sorted(wect.get((name, "random"), []), key=lambda r: r["d"])
    ax.plot([r["d"] for r in rows], [r["rel_err"]["65"] for r in rows], color=H_COLORS[65], lw=1, ls="--",
            label="random directions, $h = 65$")
    f32 = diwect.get((name, 65, "float32"))
    if f32:
        ax.axhline(f32["rel_err"], color=INK2, lw=1, ls=":", label="DIWECT in float32, $h = 65$")
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    short = {"mesh": "seed mesh", "freudenthal": "seed voxels"}.get(meta[name]["ctype"]) if meta[name]["source"] == "barley seed" else meta[name]["source"]
    ax.set_title(f"{short}\n{cells[name]:,} cells", fontsize=7, color=INK)
    ax.set_xlabel("directions $d$")
np.atleast_1d(axes)[0].set_ylabel("relative error vs. DIWECT")
h, l = np.atleast_1d(axes)[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=3, fontsize=7, bbox_to_anchor=(0.5, 1.16), handlelength=4)
fig.tight_layout()
fig.savefig(OUT / "convergence.pdf", bbox_inches="tight")
plt.show()

## Figures: time at matched accuracy

In [ ]:
# ---------------------------------------------------------------------------
H = 65
TOLS = (1e-2, 1e-3)
FAMILIES = [("grayscale image", "freudenthal", "grayscale images (2D)"),
            ("binary image", "freudenthal", "binary images (2D)"),
            ("barley seed", "mesh", "seed meshes (3D)"),
            ("barley seed", "freudenthal", "seed voxels (3D)")]
CELL_RANGE = (3e3, 6e6)


def cost_figure(diwect_seconds, wect_seconds, device, filename):
    fig, axes = plt.subplots(2, 2, figsize=(5.9, 4.2), sharey="row")
    for ax, (source, ctype, title) in zip(axes.flat, FAMILIES):
        names = sorted({n for n, _, _ in diwect if meta[n]["source"] == source and meta[n]["ctype"] == ctype},
                       key=cells.get)
        pts = [(cells[n], diwect_seconds(n)) for n in names if diwect_seconds(n) is not None]
        if not pts:
            ax.set_visible(False)
            continue
        ax.plot(*zip(*pts), color=BLUE, marker="o", ms=3, lw=1.5, label="DIWECT (exact)")
        for tol, ls in zip(TOLS, ("-", "--")):
            pts = [(cells[n], wect_seconds(n, d_for(n, H, tol)["d"])) for n in names if d_for(n, H, tol)]
            pts = [p for p in pts if p[1] is not None]
            if pts:
                ax.plot(*zip(*pts), color=ORANGE, ls=ls, marker="s", ms=3, lw=1.5,
                        mfc=ORANGE if ls == "-" else "white",
                        label=f"WECT average, error $\\leq {tol:g}$")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlim(*CELL_RANGE)
        ax.xaxis.set_major_locator(matplotlib.ticker.LogLocator(base=10, numticks=10))   # 10^4, 10^5, 10^6
        ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        ax.set_title(title, fontsize=7.5, color=INK)
    for ax in axes[1]:
        ax.set_xlabel("cells")
    for ax in axes[:, 0]:
        ax.set_ylabel(f"{device} time (s)")
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=3, fontsize=7, bbox_to_anchor=(0.5, 1.04), handlelength=4)
    fig.tight_layout()
    fig.savefig(OUT / filename, bbox_inches="tight")
    plt.show()


gpu_wect_seconds = {(n, r["d"]): r["seconds"] for (n, s), rows in wect.items() if s == "grid" for r in rows}
cost_figure(lambda n: diwect[n, H, "float64"]["seconds"], lambda n, d: gpu_wect_seconds.get((n, d)),
            "GPU" if env.get("gpu") else "CPU", "cost.pdf")

CPU_SRC = RESULTS_DIR / "results_cpu.jsonl"
cpu = None
if CPU_SRC.exists():
    cpu_recs = [json.loads(line) for line in open(CPU_SRC)]
    cpu_env = next(r for r in cpu_recs if r["kind"] == "env")
    cpu = {
        "diwect": {(r["complex"], r["h"], r["dtype"]): r for r in cpu_recs if r["kind"] == "diwect"},
        "wect": {(r["complex"], r["d"]): r for r in cpu_recs if r["kind"] == "wect" and not r.get("skipped")},
        "hcheck": [r for r in cpu_recs if r["kind"] == "wect_h_check"],
    }
    cpu_name = "CPU"
    cost_figure(lambda n: cpu["diwect"][n, H, "float64"]["seconds"] if (n, H, "float64") in cpu["diwect"] else None,
                lambda n, d: cpu["wect"][n, d]["seconds"] if (n, d) in cpu["wect"] else None,
                cpu_name, "cost_cpu.pdf")
    print(f"CPU timings from {CPU_SRC.name}: {cpu_env}")

## Tables

In [ ]:
# ---------------------------------------------------------------------------


def fmt_t(s):
    return f"{s * 1e3:.0f}\\,ms" if s < 1 else f"{s:.1f}\\,s"


def fmt_m(b):
    return f"{b / 2 ** 20:.0f}\\,MB" if b < 2 ** 30 else f"{b / 2 ** 30:.1f}\\,GB"


lines = ["% generated by 05e-efficiency-plots.ipynb from " + SRC.name,
         f"% {env.get('gpu', env.get('device'))}, torch {env['torch']}", ""]

# accuracy-matched cost
lines += ["% Table: cost at matched accuracy (h = 65)",
          "\\begin{tabular}{lrrrrrr}", "\\toprule",
          "complex & cells & \\multicolumn{2}{c}{DIWECT (float64)} & \\multicolumn{3}{c}{WECT average, error $\\leq 10^{-2}$} \\\\",
          "\\cmidrule(lr){3-4} \\cmidrule(lr){5-7}",
          " & & time & memory & $d$ & time & memory \\\\", "\\midrule"]
for n in sorted({n for n, _, _ in diwect}, key=lambda n: (meta[n]["dim"], meta[n]["source"], cells[n])):
    dw = diwect.get((n, H, "float64"))
    w = d_for(n, H, 1e-2)
    wcells = f"{w['d']:,} & {fmt_t(w['seconds'])} & {fmt_m(w['peak_bytes'])}" if w else "\\multicolumn{3}{c}{not reached}"
    lines.append(f"{label(n)} & {cells[n]:,} & {fmt_t(dw['seconds'])} & {fmt_m(dw['peak_bytes'])} & {wcells} \\\\")
lines += ["\\bottomrule", "\\end{tabular}", ""]

# cubical vs. triangulated
lines += ["% Table: cubical vs. triangulated (binary inputs), h = 65",
          "\\begin{tabular}{lrrrrrr}", "\\toprule",
          "input & cubical cells & triangulated cells & WECT, cubical & WECT, triangulated "
          "& DIWECT, cubical & DIWECT, triangulated \\\\",
          " & & & \\multicolumn{2}{c}{($d$ for error $\\leq 10^{-2}$)} & \\multicolumn{2}{c}{(float64)} \\\\",
          "\\midrule"]
for r in recs:
    if r["kind"] != "cubical_vs_freudenthal":
        continue
    c, t = r["complex"], r["complex"].replace("cubical", "freudenthal")
    wc, wt = d_for(c, H, 1e-2), d_for(t, H, 1e-2)
    dc, dt = diwect.get((c, H, "float64")), diwect.get((t, H, "float64"))
    same = "" if r["max_abs_diff"] == 0 else " (WECTs differ!)"
    lines.append(f"{label(c).split(' (')[0]}{same} & {cells[c]:,} & {cells[t]:,} & "
                 f"{fmt_t(wc['seconds']) if wc else '--'} & {fmt_t(wt['seconds']) if wt else '--'} & "
                 f"{fmt_t(dc['seconds']) if dc else '--'} & {fmt_t(dt['seconds']) if dt else '--'} \\\\")
lines += ["\\bottomrule", "\\end{tabular}", ""]

# dependence on the number of heights
lines += ["% Table: dependence on the number of heights h",
          "\\begin{tabular}{lr" + "r" * len(HEIGHTS) * 2 + "}", "\\toprule",
          "complex & cells & " + " & ".join(f"DIWECT $h={h}$" for h in HEIGHTS) + " & "
          + " & ".join(f"$d$ for $10^{{-2}}$, $h={h}$" for h in HEIGHTS) + " \\\\", "\\midrule"]
for n in sorted({n for n, _, _ in diwect}, key=lambda n: (meta[n]["dim"], meta[n]["source"], cells[n])):
    t = [fmt_t(diwect[n, h, "float64"]["seconds"]) if (n, h, "float64") in diwect else "--" for h in HEIGHTS]
    d = [f"{d_for(n, h, 1e-2)['d']:,}" if d_for(n, h, 1e-2) else "--" for h in HEIGHTS]
    lines.append(f"{label(n)} & {cells[n]:,} & " + " & ".join(t + d) + " \\\\")
lines += ["\\bottomrule", "\\end{tabular}", ""]

hcheck = {}
for r in recs:
    if r["kind"] == "wect_h_check":
        hcheck.setdefault(r["complex"], {})[r["h"]] = r["seconds"]
ratios = [v[257] / v[17] for v in hcheck.values() if 17 in v and 257 in v]
lines.append(f"% WECT time ratio h=257 / h=17 at d=1024: median {np.median(ratios):.2f}, "
             f"range {min(ratios):.2f}-{max(ratios):.2f}")
open(OUT / "tables.tex", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))

# CPU vs. GPU at matched accuracy
if cpu:
    lines = ["% generated by 05e-efficiency-plots.ipynb from " + CPU_SRC.name,
             f"% CPU: {cpu_env['cpu']}, {cpu_env['cpu_count']} vCPU, {cpu_env['threads']} threads, torch {cpu_env['torch']}",
             "", "% Table: CPU vs. GPU time at matched accuracy (h = 65)",
             "\\begin{tabular}{lrrrrrrr}", "\\toprule",
             "complex & cells & \\multicolumn{3}{c}{DIWECT (float64)} & \\multicolumn{3}{c}{WECT average, error $\\leq 10^{-2}$} \\\\",
             "\\cmidrule(lr){3-5} \\cmidrule(lr){6-8}",
             " & & CPU & GPU & CPU memory & $d$ & CPU & GPU \\\\", "\\midrule"]
    for n in sorted({n for n, _, _ in diwect}, key=lambda n: (meta[n]["dim"], meta[n]["source"], cells[n])):
        c, g = cpu["diwect"].get((n, H, "float64")), diwect.get((n, H, "float64"))
        w = d_for(n, H, 1e-2)
        cw = cpu["wect"].get((n, w["d"])) if w else None
        dcol = f"{fmt_t(c['seconds'])} & {fmt_t(g['seconds'])} & {fmt_m(c['peak_rss_bytes'])}" if c else "-- & -- & --"
        wcol = (f"{w['d']:,} & {fmt_t(cw['seconds']) if cw else '--'} & {fmt_t(w['seconds'])}" if w
                else "\\multicolumn{3}{c}{not reached}")
        lines.append(f"{label(n)} & {cells[n]:,} & {dcol} & {wcol} \\\\")
    lines += ["\\bottomrule", "\\end{tabular}", ""]
    errs = [r["rel_err"] for r in cpu["diwect"].values() if r["dtype"] == "float64"]
    lines.append(f"% CPU float64 DIWECT vs. GPU reference: max relative difference {max(errs):.1e}")
    hc = {}
    for r in cpu["hcheck"]:
        hc.setdefault(r["complex"], {})[r["h"]] = r["seconds"]
    ratios = [v[257] / v[17] for v in hc.values() if 17 in v and 257 in v]
    if ratios:
        lines.append(f"% CPU WECT time ratio h=257 / h=17 at d=64: median {np.median(ratios):.2f}, "
                     f"range {min(ratios):.2f}-{max(ratios):.2f}")
    open(OUT / "tables_cpu.tex", "w").write("\n".join(lines) + "\n")
    print("\n".join(lines))